In [89]:
# Cell 1 - Imports
import os
from dotenv import load_dotenv
from llama_index.core import StorageContext, load_index_from_storage
from pymilvus import MilvusClient, utility
import numpy as np
from tqdm import tqdm

load_dotenv()

True

In [90]:
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding
embed_model = OllamaEmbedding(model_name="nomic-embed-text:v1.5")
Settings.embed_model = embed_model

In [91]:
# Cell 2 - Load existing vector store and setup paths
PERSIST_DIR = "storage"  # For the existing vector store
MILVUS_DB = "ardania_vec_store.db"  # Milvus DB in root directory

def load_existing_store():
    """Load the existing vector store from disk"""
    if not os.path.exists(PERSIST_DIR):
        raise ValueError(f"Storage directory '{PERSIST_DIR}' not found")
    
    print("Loading existing vector store...")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)
    print("Vector store loaded successfully")
    return index

In [92]:
# Cell 3 - Initialize Milvus Lite connection
def init_milvus():
    """Initialize Milvus Lite with local file"""
    print("Initializing Milvus Lite...")
    try:
        db_path = "./milvus_ardania.db"
        client = MilvusClient(db_path)
        print(f"Milvus Lite connected successfully using: {db_path}")
        return client
    except Exception as e:
        print(f"Failed to connect to Milvus Lite: {e}")
        raise

In [93]:
# Cell 4 - Create Milvus collection
def create_milvus_collection(client, dimension=1536):
    """Create a new Milvus collection for the vector store"""
    collection_name = "ardania_vectors"
    
    # Drop existing collection if it exists
    if client.get_collection(collection_name):
        client.drop_collection(collection_name)
        print(f"Dropped existing collection: {collection_name}")
    
    # Create collection with schema
    schema = {
        "id": {"dtype": "varchar", "max_length": 100, "primary_key": True},
        "text": {"dtype": "varchar", "max_length": 65535},
        "metadata": {"dtype": "json", "max_length": 65535},
        "vector": {"dtype": "float_vector", "dim": dimension}
    }
    
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params={
            "vector": {
                "metric_type": "L2",
                "index_type": "FLAT"
            }
        }
    )
    
    print(f"Created Milvus collection: {collection_name}")
    return collection_name

In [94]:
# Cell 5 - Migrate data to Milvus
def migrate_to_milvus(index, client):
    """Migrate existing vector store data to Milvus"""
    # Create collection
    collection_name = create_milvus_collection(client)
    
    # Get all nodes from the existing index
    all_nodes = list(index.docstore.docs.values())
    
    # Prepare data for batch insert
    entities = []
    print("Preparing data for migration...")
    for node in tqdm(all_nodes):
        if hasattr(node, 'embedding'):
            entity = {
                "id": node.node_id,
                "text": node.text,
                "metadata": node.metadata if hasattr(node, 'metadata') else {},
                "vector": node.embedding
            }
            entities.append(entity)
    
    # Insert data in batches
    batch_size = 1000
    total_docs = len(entities)
    
    print("Inserting documents into Milvus...")
    for i in tqdm(range(0, total_docs, batch_size)):
        batch = entities[i:i + batch_size]
        client.insert(collection_name, batch)
    
    print(f"Successfully migrated {total_docs} documents to Milvus")
    return collection_name

In [95]:
# Cell 6 - Main migration process
def main():
    try:
        # Load existing store
        index = load_existing_store()
        
        # Initialize Milvus Lite
        client = init_milvus()
        
        # Perform migration
        collection_name = migrate_to_milvus(index, client)
        
        print("Vector store successfully migrated to Milvus Lite")
        return client, collection_name
    
    except Exception as e:
        print(f"Error during migration: {e}")
        raise

In [96]:
# Cell 7 - Run migration
if __name__ == "__main__":
    new_index = main()

Loading existing vector store...
Loading llama_index.core.storage.kvstore.simple_kvstore from storage\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from storage\index_store.json.


2025-08-13 03:31:59,964 [ERROR][create_connection]: Failed to create new connection using: ./milvus_ardania.db (_utils.py:48)


Vector store loaded successfully
Initializing Milvus Lite...
Failed to connect to Milvus Lite: No module named 'milvus_lite'
Error during migration: No module named 'milvus_lite'


ModuleNotFoundError: No module named 'milvus_lite'